In [1]:
import numpy as np
import jax
import jaxlib
import jax.numpy as jnp
import flax
import flax.linen as nn
import optax
from typing import Tuple, Callable, Any, Dict, Optional
import numpy.typing as npt
import copy
import pathlib
import matplotlib.pyplot as plt
import time
import json
import ast
import netket as nk
import os
import glob
import sys
sys.path.append("/home/ihuarte/Escritorio/Ivan/NNs")

#os.chdir("/home/ihuarte/Escritorio/Ivan/NN/")

# from ATMOS_VA.VA_project.src.VA_project.model.model import OxalateJKGamma
# from ATMOS_VA.VA_project.src.VA_project.engine.runners import Runner

#from NN_utils import load_vstate
#from correlations import correlations_vstate

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "gpu")
print(jax.devices())

from NN_module.ST_utils import compare_params, masked_optimizer



[CudaDevice(id=0)]


# TO DO
- Inspeccionar que init_model_auto pasa bien el diccionario con los formatos correctos
- Compatibilizar todo para modelos simples
- Terminar de integrar todo cuando furule

In [2]:
def make_hashable(obj):
    if isinstance(obj, dict):
        return frozenset((k, make_hashable(v)) for k, v in obj.items())
    elif isinstance(obj, (list, tuple)):
        return tuple(make_hashable(v) for v in obj)
    elif isinstance(obj, set):
        return frozenset(make_hashable(v) for v in obj)
    else:
        return obj

In [3]:
a = {

    "module": "SplitTraining",
    "setup":{

        "modulus_setup":{
            "module": "MotherModule",
            "setup":{
                "Core":{
                    "module":"CvT3",
                    "setup":{
                        "n_CP_blocks": [2, 2],
                        "CTemb_channels": [32, 16],
                        "CP_channels": [32, 16],
                        "attn_heads": [4, 4],
                        "kernel": [3, 3],
                        "final_architecture": [128,5]
                    }
                },
                "Final":{
                    "module": None,
                    "setup":{}
                }
            }
        },

        "phase_setup":{
            "module": "MotherModule",
            "setup":{
                "Core":{
                    "module":"CvT3",
                    "setup":{
                        "n_CP_blocks": [2, 2],
                        "CTemb_channels": [32, 16],
                        "CP_channels": [32, 16],
                        "attn_heads": [4, 4],
                        "kernel": [3, 3],
                        "final_architecture": [128,5]
                    }
                },
                "Final":{
                    "module": None,
                    "setup":{}
                }
            }    
        }
    },

    "symm_Z2": True,
    "trivial_Z2": True,
    "symm_2D": True

    
}

from NN_module.initialize_models import FactoryBuilder

FactoryBuilder(a, **{'lattice_size':[4,4]})

{'lattice_size': [4, 4]}


AttributeError: 'NoneType' object has no attribute 'items'

In [21]:
from typing import Callable, Sequence, Tuple, Any
from NN_module.NN_utils import traslations_2D

class SplitTraining_Worker(nn.Module):
    """
    Flax module to train module and phase separately
    """

    modulus_clss: nn.Module
    phase_clss: nn.Module

    modulus_setup: dict        # Frozen and hashable dict
    phase_setup: dict        # Frozen and hashable dict

    def setup(self):
        self.mod_model = self.modulus_clss(**self.modulus_setup)
        self.ph_model = self.phase_clss(**self.phase_setup)

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        print(f"x_in.shape: {x.shape}")

        log_modulus = self.mod_model(x)
        phase = self.ph_model(x)

        return log_modulus + 1j * phase
    
class SplitTraining_2D(nn.Module):

    modulus_clss: nn.Module
    phase_clss: nn.Module

    modulus_setup: dict        # Frozen and hashable dict
    phase_setup: dict        # Frozen and hashable dict

    lattice_size: Tuple[int, int] = None
    token_size: Tuple[int, int] = None

    @nn.compact
    def __call__(self, x):

        worker = SplitTraining_Worker(
            modulus_clss=self.modulus_clss,
            phase_clss=self.phase_clss,
            modulus_setup=self.modulus_setup,
            phase_setup=self.phase_setup
        )

        # 2D traslation
        print(f"x_before: {x.shape}")
        traslational_x = traslations_2D(  
            x, size=self.lattice_size, token_size=self.token_size, memory=False
        )
        print(f"Translational x shape: {traslational_x.shape}")

        return jax.vmap(worker, in_axes=0)(traslational_x).mean(axis=0)

class SplitTraining_Z2(nn.Module):

    modulus_clss: nn.Module
    phase_clss: nn.Module

    modulus_setup: dict        # Frozen and hashable dict
    phase_setup: dict        # Frozen and hashable dict

    lattice_size: Tuple[int, int] = None
    token_size: Tuple[int, int] = None

    "Symmetries"
    symm_2D: bool = False
    trivial_Z2: bool = False

    @nn.compact
    def __call__(self, x):

        if self.symm_2D:
            worker = SplitTraining_2D(
                modulus_clss=self.modulus_clss,
                phase_clss=self.phase_clss,
                modulus_setup=self.modulus_setup,
                phase_setup=self.phase_setup,
                lattice_size=self.lattice_size,
                token_size=self.token_size
            )
        else:
            worker = SplitTraining_Worker(
                modulus_clss=self.modulus_clss,
                phase_clss=self.phase_clss,
                modulus_setup=self.modulus_setup,
                phase_setup=self.phase_setup
            )

        output_x = jnp.atleast_1d(worker(x))
        output_inv_x = jnp.atleast_1d(worker(-x))

        # Concatenamos las dos contribuciones
        z2_stack = jnp.stack([output_x, output_inv_x], axis=0)

        if self.trivial_Z2:
            res = jax.nn.logsumexp(z2_stack, axis=0)
            return res
        else:
            b = jnp.array([1.0, -1.0])[:, None]
            res = jax.nn.logsumexp(z2_stack, b=b, axis=0)
            return res


class SplitTraining(nn.Module):
    """
    Flax module to train module and phase separately
    """

    modulus_clss: nn.Module
    phase_clss: nn.Module
    
    modulus_setup: dict        # Frozen and hashable dict
    phase_setup: dict        # Frozen and hashable dict


    "Symmetries"
    symm_Z2: bool = False
    symm_2D: bool = False
    trivial_Z2: bool = True

    "Needed for performing 2D traslation symmetries"
    lattice_size: Tuple[int, int] = None
    token_size: Tuple[int, int] = None

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:

        if self.symm_Z2:
            worker = SplitTraining_Z2(
                modulus_clss=self.modulus_clss,
                phase_clss=self.phase_clss,
                modulus_setup=self.modulus_setup,
                phase_setup=self.phase_setup,

                trivial_Z2=self.trivial_Z2,
                symm_2D=self.symm_2D,
                lattice_size=self.lattice_size,
                token_size=self.token_size
                
            )
        elif self.symm_2D:
            worker = SplitTraining_2D(
                modulus_clss=self.modulus_clss,
                phase_clss=self.phase_clss,
                modulus_setup=self.modulus_setup,
                phase_setup=self.phase_setup,

                lattice_size=self.lattice_size,
                token_size=self.token_size
            )
        else:
            worker = SplitTraining_Worker(
                modulus_clss=self.modulus_clss,
                phase_clss=self.phase_clss,
                modulus_setup=self.modulus_setup,
                phase_setup=self.phase_setup
            )

        x = worker(x)

        return x


In [22]:
from NN_module.NN_utils import activation_dict
from frozendict import deepfreeze
import importlib

def recursive_list_to_tuple(target):

    out = []
    if all(isinstance(element, list) for element in target):
        for element in target:
            out.append(recursive_list_to_tuple(element))
            
    else:
        return tuple(target)    

    return tuple(out)

def get_model_class(path):

    module_path, clss = path.split(":")
    module = importlib.import_module(module_path)
    model = getattr(module, clss)

    return model

def preprocess_setup(setup: dict) -> dict:

    clean_setup = {}

    for k, v in setup.items():

        if isinstance(v, dict):
            v = preprocess_setup(v)
            
        elif isinstance(v, list):
            v = recursive_list_to_tuple(v)

        elif 'activation' in k:
            if all([type(act) in [str, int] for act in v]):
                v = tuple(
                    [activation_dict[act] if act != 0 else 0 for act in v]
                )

        clean_setup[k] = v

    return clean_setup

def init_model_auto(model_setup):

    model_class = get_model_class(model_setup["path"])
    adapted_setup = preprocess_setup(model_setup['setup'])

    return model_class(**adapted_setup)

def _init_model(name, model_setup, split_training=True):

    if split_training:

        model_setup["modulus_setup"]['setup']['name'] = 'modulus'
        model_setup["phase_setup"]['setup']['name'] = 'phase'

        model_setup["modulus_setup"]['setup']['lattice_size'] = model_setup['lattice_size']  
        model_setup["phase_setup"]['setup']['lattice_size'] = model_setup['lattice_size']  

        lattice_size = tuple(model_setup["lattice_size"]) if model_setup["symm_2D"] else None
        print(f"lattice_size: {lattice_size}")
        token_size = tuple(
                model_setup["modulus_setup"]["setup"]["token_size"]
            ) if 'ViT' in name else None
        
        #model_setup = preprocess_setup(model_setup)
        
        modulus_clss = get_model_class(model_setup["modulus_setup"]["path"])
        phase_clss = get_model_class(model_setup["phase_setup"]["path"])

        frozen_modulus_setup = deepfreeze(model_setup["modulus_setup"]["setup"])
        frozen_phase_setup = deepfreeze(model_setup["phase_setup"]["setup"])
 
        return SplitTraining(
            modulus_clss=modulus_clss,
            phase_clss=phase_clss,
            modulus_setup=frozen_modulus_setup,
            phase_setup=frozen_phase_setup,
            symm_2D=model_setup["symm_2D"],
            symm_Z2=model_setup["symm_Z2"],
            trivial_Z2=model_setup["trivial_Z2"],
            lattice_size=lattice_size,
            token_size=token_size
            
        )
    

model_setup = {

    "modulus_setup":{
        "path": "NN_module.models.CvT3:CvT3",
        "setup":{
            "n_CP_blocks": [2, 2],
            "CTemb_channels": [32, 16],
            "CP_channels": [32, 16],
            "attn_heads": [4, 4],
            "kernel": [3, 3],
            "final_architecture": "(128,5,)"
        }
    },

    "phase_setup":{
        "path": "NN_module.models.Phase:CNNClsf",
        "setup":{
            "cnnclsf_channels": [32],
            "n_classes": 2
        }
    },

    "symm_Z2": True,
    "trivial_Z2": True,
    "symm_2D": True
}

deepfreeze(model_setup)

frozendict.frozendict({'modulus_setup': frozendict.frozendict({'path': 'NN_module.models.CvT3:CvT3', 'setup': frozendict.frozendict({'n_CP_blocks': (2, 2), 'CTemb_channels': (32, 16), 'CP_channels': (32, 16), 'attn_heads': (4, 4), 'kernel': (3, 3), 'final_architecture': '(128,5,)'})}), 'phase_setup': frozendict.frozendict({'path': 'NN_module.models.Phase:CNNClsf', 'setup': frozendict.frozendict({'cnnclsf_channels': (32,), 'n_classes': 2})}), 'symm_Z2': True, 'trivial_Z2': True, 'symm_2D': True})

In [23]:
import numpy as np
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
from netket.operator.spin import sigmaz
import optax
import json
import time
import argparse
import ast
import os

# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "gpu")
jax.devices()

# Añadir los directorios necesarios
import sys
from pathlib import Path

sys.path.append("/home/ihuarte/Escritorio/Ivan/ATMOS_VA/VA_project/src")
sys.path.append(
    "/home/ihuarte/Escritorio/Ivan/Transformers/transformer_LR_WF_public"
)
# Importar módulos necesarios
from VA_project.model.model import J1J2Square
from VA_project.engine.runners import Runner
from NN_module.sim_utils import save_results, dump_callback, init_model, BestIterKeeper
from NN_module.label_utils import (
    get_filenames_from_settings,
    architecture_label,
    get_write_folder_from_model,
    display_simulation_settings,
    get_ST_folder,
)
from NN_module.schedules import get_ST_schedule
from NN_module.NN_utils import scheduler_initializer, phase_stats_vstate, modphase
from NN_module.ST_utils import check_zero_grads, compare_params, masked_optimizer
from NN_module.observables import calc_all_observables_vs, calc_all_observables_ED
from transformer_LR_WF.utils import InvertMagnetization

configurations = [
        "/home/ihuarte/Escritorio/Ivan/NNs/config.json",
        "/home/ihuarte/Escritorio/Ivan/NNs/config_CM.json",
        "/home/ihuarte/Escritorio/Ivan/NNs/config_NN.json",
    ]

# Cargamos configuraciones de archivos json
with open(configurations[0], "r") as f:
    config = json.load(f)
with open(configurations[1], "r") as f:
    config_cm = json.load(f)
with open(configurations[2], "r") as f:
    config_nn = json.load(f)

cm_model_name = config_cm["CM"]["selection"]
nn_model_name = config_nn["model_NN"]["selection"]

nn_model_setup = config_nn["model_NN"][nn_model_name]
model_label = cm_model_name + "_" + nn_model_name

sizes = config_cm["sizes"]
J1_list = config_cm["CM"][cm_model_name]["J1_list"]  # Lattice and coupling model
J2_list = config_cm["CM"][cm_model_name]["J2_list"]  # Lattice and coupling model
fields_list = config_cm["CM"][cm_model_name]["fields_list"]
kwargs_lattice = config_cm["kwargs_lattice"]

# Simulation settings
split_training = config["split_training"]
training_name = "ST_schedule" if split_training else "normal_schedule"
lr_name = config[training_name]["lr_name"]
training_setup = config[training_name]["setup"]
lr_schedule_setup = config[training_name]["lr_schedules"][lr_name]

exact_diag = config["exact_diagonalization"]
dump_simulation = config["dump_sim_callback"]

sampler_setup = config["sampler"]
n_samples = (
    sampler_setup["n_samples_per_chain"]
    * sampler_setup["n_chains_per_rank"]
    * sampler_setup["n_ranks"]
)
print(f"Total samples: {n_samples}")

write = get_write_folder_from_model({**config, **config_cm, **config_nn})

### MC sampling rules ###
rule1 = nk.sampler.rules.LocalRule()
rule2 = InvertMagnetization()
pinvert = 0.25
pflip = 1 - pinvert

E_ED = None
x_ED = None


model_setup = {

    "modulus_setup":{
        "path": "NN_module.models.CvT3:CvT3",
        "setup":{
            "n_CP_blocks": [2, 2],
            "CTemb_channels": [32, 16],
            "CP_channels": [32, 16],
            "attn_heads": [4, 4],
            "kernel": [3, 3],
            "final_architecture": [128,5]
        }
    },

    "phase_setup":{
        "path": "NN_module.models.Phase:CNNClsf",
        "setup":{
            "cnnclsf_channels": [32],
            "n_classes": 2
        }
    },

    "symm_Z2": True,
    "trivial_Z2": True,
    "symm_2D": True
}


for i, size in enumerate(sizes):
    print(size)
    model_setup["lattice_size"] = size

    N = int(np.prod(size))
    if N > 20:
        exact_diag = False

    write_folder_size = write + f"Size_{size[0]}x{size[1]}/"

    training_folder = get_ST_folder(split_training, training_setup)

    write_folder = write_folder_size + f"{training_folder}/"

    ###  Reseting Hilbert space object and the observables ###
    hi = nk.hilbert.Spin(s=1 / 2, N=N)

    ## Reset sampler
    sampler = nk.sampler.MetropolisSampler(
        hi,
        nk.sampler.rules.MultipleRules([rule1, rule2], [pflip, pinvert]),
        n_chains_per_rank=sampler_setup["n_chains_per_rank"],
        chunk_size=sampler_setup["chunk_sampler"],
    )

    for fields in fields_list:

        for J1, J2 in zip(J1_list, J2_list):

            config_cm["CM"][cm_model_name]["J1"] = J1
            config_cm["CM"][cm_model_name]["J2"] = J2
            config_cm["CM"][cm_model_name]["fields"] = fields
            config_cm["size"] = size
            display_simulation_settings({**config_cm, **config_nn})

            ## Update Hamiltonian

            j1j2 = J1J2Square(size, J1, J2, fields, **kwargs_lattice)

            eng = Runner(j1j2.cm)
            H = eng.build_hamiltonian()

            if exact_diag:
                print("Running exact diagonalization...")
                E_ED, x_ED = eng.exact_energy_lanczos(eigenstates=True)
                E_ED = float(E_ED.squeeze(-1))
                print(f"Energy ED: {E_ED}")

            ####################################################

            callback_artifacts = {}
            time_in = time.time()

            model = _init_model(nn_model_name, model_setup, split_training)

            log = (
                nk.logging.RuntimeLog()
            )  # If instead of this logging you insert a string, it will be used as output prefix for a JSON file where the evolution of the energy at each epoch will be stored.

            # Initialize vstate with parameters
            vstate = nk.vqs.MCState(
                sampler,
                model=model,
                n_samples=n_samples,
                n_discard_per_chain=0,
                chunk_size=sampler_setup["chunk_vstate"],
            )

            if split_training:  # Alternated training between modulus and phase

                segments = []
                modes = []
                s, m, r = (
                    training_setup["segments"],
                    training_setup["mode"],
                    training_setup["repeat_segment"],
                )

                for seg, mode, repeats in zip(s, m, r):
                    segments += [seg] * repeats
                    modes += [mode] * repeats

                total_segments = len(segments)
                total_epochs = int(np.array([s for seg in segments for s in seg]).sum())
                training_setup["total_epochs"] = total_epochs

                lr_schedule = get_ST_schedule(
                    lr_name,
                    {
                        **lr_schedule_setup,
                        "total_epochs": total_epochs,
                        "total_segments": total_segments,
                    },
                )
                ds_schedule = jnp.linspace(1e-2, 1e-4, total_segments)

                transformations = {
                    "train": optax.sgd(0.1),
                    "freeze": optax.set_to_zero(),
                }

                keeper = BestIterKeeper(
                    total_epochs, H, N, baseline=1e-8, mode="best_energy"
                )
                # keeper.filename = 'Somewhere' #It allows you to store the parameters of the model for the state with lowest energy found.

                print(f"\nEpochs:      {total_epochs}")
                print(f"Segments:      {segments}")
                print(f"Modes:         {modes}")

                for i, (segment, seg_modes, lr) in enumerate(
                    zip(segments, modes, lr_schedule)
                ):

                    print(
                        f"\nSegment {i+1} of {total_segments}......   lr: {lr:.4f}  ds: {ds_schedule[i]:.4f}\n"
                    )
                    SR = nk.optimizer.SR(diag_shift=ds_schedule[i])

                    for epochs, mode in zip(segment, seg_modes):

                        # P0 = vstate.parameters

                        if mode == "M":
                            mode = "modulus"
                            mask = "phase"
                            transformations["train"] = optax.sgd(learning_rate=lr)

                        elif mode == "P":
                            mode = "phase"
                            mask = "modulus"
                            transformations["train"] = optax.sgd(learning_rate=lr)

                        elif mode == "B":
                            mode = "both"
                            mask = None
                            transformations["train"] = optax.sgd(learning_rate=lr)

                        else:
                            raise ValueError(
                                f"Invalid training mode: {mode}."
                                f"'M' for modulus and 'P' for phase"
                            )

                        variables = vstate.variables
                        sampler = vstate.sampler
                        optimizer, _ = masked_optimizer(
                            vstate.parameters, transformations, mode=mask
                        )

                        vstate = nk.vqs.MCState(
                            sampler,
                            sampler_seed=vstate.sampler_state.rng,
                            model=model,
                            n_samples=n_samples,
                            n_discard_per_chain=0,
                            chunk_size=sampler_setup["chunk_vstate"],
                            variables=variables,
                        )

                        gs = nk.driver.VMC(
                            H,
                            optimizer,
                            variational_state=vstate,
                            preconditioner=SR,
                        )

                        print(f"\nTraining {mode} for {epochs} epochs...")
                        gs.run(
                            n_iter=epochs,
                            out=log,
                            callback=[keeper.update],
                            show_progress=True,
                        )
                        mean, std, psi = phase_stats_vstate(vstate)
                        print(f"VS phase: {mean} \u00b1 {std}  ({psi})")

                        # P1 = vstate.parameters
                        # print(compare_params(P0, P1))
                        # check_zero_grads(vstate, mask)

            else:  # Training modulus and phase at the same time

                total_epochs = training_setup["total_epochs"]
                keeper = BestIterKeeper(
                    total_epochs, H, N, baseline=1e-8, mode="best_energy"
                )
                # keeper.filename = 'Somewhere' #It allows you to store the parameters of the model for the state with lowest energy found.
                lr_schedule_setup["total_epochs"] = total_epochs

                ds_schedule = optax.linear_schedule(1e-2, 1e-4, total_epochs)
                SR = nk.optimizer.SR(diag_shift=ds_schedule)
                lr_schedule = scheduler_initializer(
                    "warmup_exponential_decay", lr_schedule_setup
                )
                optimizer = nk.optimizer.Sgd(learning_rate=lr_schedule)
                gs = nk.driver.VMC(
                    H, optimizer, variational_state=vstate, preconditioner=SR
                ).run(
                    n_iter=total_epochs,
                    out=log,
                    callback=[keeper.update],
                    show_progress=True,
                )


Total samples: 1024
[4, 4]


────────────────────────────── 🧲 CM: J1J2Square   🧠 NN_architecture: CvT3_CNNClsf ───────────────────────────────

🧱 Size: 4x4

╭────────────────────────────────────────────────── J1J2Square ───────────────────────────────────────────────────╮
│ J1= 1.0  J2= 0.5  fields= [0.0, 0.0, 0.0]                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── CvT3_CNNClsf ──────────────────────────────────────────────────╮
│ n_CP_blocks= [2, 2]        CTemb_channels= [32,     CP_channels= [32, 16]  attn_heads= [4, 4]  kernel= [3, 3]   │
│                            16]                                                                                  │
│ final_architecture=        cnnclsf_channels= [32]   n_classes= 2           symm_Z2= True       trivial_Z2= True │
│ (128,5,)                                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

x_before: (16, 16)
Translational x shape: (16, 16, 16)
x_in.shape: (16, 16)
x_before: (16, 16)
Translational x shape: (16, 16, 16)
x_in.shape: (16, 16)
x_before: (16, 16)
Translational x shape: (16, 16, 16)
x_in.shape: (16, 16)
x_before: (16, 16)
Translational x shape: (16, 16, 16)
x_in.shape: (16, 16)
x_before: (10, 16)
Translational x shape: (16, 10, 16)
x_in.shape: (10, 16)
x_before: (10, 16)
Translational x shape: (16, 10, 16)
x_in.shape: (10, 16)
x_before: (490, 16)
Translational x shape: (16, 490, 16)
x_in.shape: (490, 16)
x_before: (490, 16)
Translational x shape: (16, 490, 16)
x_in.shape: (490, 16)
x_before: (10, 16)
Translational x shape: (16, 10, 16)
x_in.shape: (10, 16)
x_before: (10, 16)
Translational x shape: (16, 10, 16)
x_in.shape: (10, 16)
x_before: (490, 16)
Translational x shape: (16, 490, 16)
x_in.shape: (490, 16)
x_before: (490, 16)
Translational x shape: (16, 490, 16)
x_in.shape: (490, 16)
x_before: (4, 16)
Translational x shape: (16, 4, 16)
x_in.shape: (4, 16)
x_b

2025-10-23 15:56:13.681074: E external/xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng20{k2=7,k3=0} for conv %cudnn-conv-bw-filter.216 = (f64[32,1,3,3]{3,2,1,0}, u8[0]{0}) custom-call(%bitcast.196612, %bitcast.201379), window={size=3x3}, dim_labels=bf01_oi01->bf01, feature_group_count=32, custom_call_target="__cudnn$convBackwardFilter", metadata={op_name="jit(forces_expect_hermitian_chunked)/jit(main)/jit(shmap_body)/while/body/transpose(jvp(SplitTraining))/SplitTraining_Z2_0/SplitTraining_2D_0/vmap(SplitTraining_Worker_0)/modulus/CvTWorker_0/StageBlock_0/ConvProjectionBlock_0/DepthPointwiseConv_0/Conv_0/conv_general_dilated" source_file="/home/ihuarte/Escritorio/Ivan/NNs/NN_module/models/CvT3.py" source_line=126}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2025-10-23 15:

x_before: (1024, 16)
Translational x shape: (16, 1024, 16)
x_in.shape: (1024, 16)
x_before: (1024, 16)
Translational x shape: (16, 1024, 16)
x_in.shape: (1024, 16)
x_before: (1, 16)
Translational x shape: (16, 16)
x_in.shape: (16,)
x_before: (1, 16)
Translational x shape: (16, 16)
x_in.shape: (16,)
x_before: (1, 16)
Translational x shape: (16, 16)
x_in.shape: (16,)
x_before: (1, 16)
Translational x shape: (16, 16)
x_in.shape: (16,)
x_before: (10, 16)
Translational x shape: (16, 10, 16)
x_in.shape: (10, 16)
x_before: (10, 16)
Translational x shape: (16, 10, 16)
x_in.shape: (10, 16)
x_before: (490, 16)
Translational x shape: (16, 490, 16)
x_in.shape: (490, 16)
x_before: (490, 16)
Translational x shape: (16, 490, 16)
x_in.shape: (490, 16)
x_before: (10, 16)
Translational x shape: (16, 10, 16)
x_in.shape: (10, 16)
x_before: (10, 16)
Translational x shape: (16, 10, 16)
x_in.shape: (10, 16)
x_before: (490, 16)
Translational x shape: (16, 490, 16)
x_in.shape: (490, 16)
x_before: (490, 16)
Tra

2025-10-23 15:58:31.468911: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 6.79GiB (rounded to 7289222912)requested by op 
2025-10-23 15:58:31.469154: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] *___******__________________________________________________________________________________________
E1023 15:58:31.469190   86686 pjrt_stream_executor_client.cc:3077] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 7289222808 bytes. [tf-allocator-allocation-error='']


ValueError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 7289222808 bytes.

In [ ]:
from NN_module.NN_utils import modphase
import itertools

def all_spin_configurations(N):
    # Genera todas las combinaciones posibles de N spines con valores ±1
    return jnp.array(list(itertools.product([-1, 1], repeat=N)))

def print_max_contributors(x, size, N_max=10):
    (mod, ph), _ = modphase(x)
    configs = all_spin_configurations(size[0]*size[1])

    idx = jnp.argsort(mod)[::-1][:N_max]
    max_configs = configs[idx,:]
    max_mods = mod[idx]
    max_phs = ph[idx]


    for config, mod, phs in zip(max_configs, max_mods, max_phs):
        print(f"Config: \n{config.reshape(size)}")
        print(f"\nModulus: {mod}")
        print(f"Phase: {phs}\n")


In [ ]:
print_max_contributors(x_ED, size=(4,4), N_max=20)

In [ ]:
ize=(4,4)
N_max=5
(mod, ph), _ = modphase(x_ED)
configs = all_spin_configurations(size[0]*size[1])

idx = jnp.argsort(mod)[::-N_max]
max_configs = configs[idx,:]
max_mods = mod[idx]
max_phs = ph[idx]

In [ ]:
idx = jnp.argsort(mod)[::-N_max]

In [ ]:
def logpsi(p, s):
    return apply_fun({"params": p}, s)

params = vstate.parameters
apply_fun = vstate._apply_fun
s_batch = vstate.samples.reshape(-1, vstate.hilbert.size)
s_batch = jnp.asarray(s_batch, dtype=jnp.complex128)
grad_logpsi = jax.grad(lambda p, s: logpsi(p, s), holomorphic=True)

#y, Jt = jax.vjp(lambda p, s: logpsi(p, s),params,s_batch)

grads = jax.vmap(lambda s: grad_logpsi(params, s))(s_batch)

In [ ]:
Jt(y)

In [ ]:
P0 = vs_0.parameters
P1 = vs_1.parameters

G0 = jax.vmap(lambda s: grad_logpsi(P0, s))(s_batch)
G1 = jax.vmap(lambda s: grad_logpsi(P1, s))(s_batch)

compare_params(G0, G1, atol=1e-20)

In [ ]:
transformations = {
    "train": optax.sgd(0.1),
    "freeze": optax.set_to_zero(),
}

grad_trans, tree_mask = masked_optimizer(P0, transformations, 'modulus')


In [ ]:
nk.jax.tree_cast()

In [ ]:
a = np.array([0, np.pi])
a.std()

In [ ]:
def energy(params, vstate):
    return vstate.expect(H).mean.real
grads = jax.grad(energy)(vstate.parameters, vstate)

In [ ]:
grads

In [ ]:
import sys
from pathlib import Path
sys.path.append(Path().resolve().parent.parent / "ATMOS_VA/VA_project/src")
from VA_project.model.model import J1J2Square
from VA_project.model.cm import HeisenbergXYZ
from VA_project.engine.runners import Runner

from NN_module.NN_utils import modphase

size=(4,4)
fields=[0.0,0.0,0.0]
kwargs_lattice={'bc':'periodic', 'order':'default_2'}
J1_list=np.linspace(-1,1,5)
J2_list=np.linspace(-1,1,5)

for J1 in J1_list:
    for J2 in J2_list:
        if J1==0.0 and J2==0.0:
            continue
        j1j2 = J1J2Square(size, J1, J2, fields, **kwargs_lattice)
        eng=Runner(j1j2.cm, S_operators=True)
        E_lanc = eng.exact_energy_lanczos(k = 100, eigenstates=False)

        fig, ax = plt.subplots()
        ax.set_title(f"J1-J2 model   J1:{J1}  J2:{J2}   size:{size[0]}x{size[1]}")
        ax.plot(range(len(E_lanc)), np.sort(E_lanc), marker='o', ms=1.2, color='r', label='Energy')
        ax.set_xlabel("Sorted eigenvalues")
        ax.set_ylabel(f"Energy")
        ax.grid()
        plt.tight_layout()
        plt.savefig(f"/home/ihuarte/Escritorio/Ivan/NNs/J1J2Spectra/Spctra_size_{size[0]}x{size[1]}_J1_{J1}_J2_{J2}.png",  dpi=600, bbox_inches="tight")


In [ ]:
idx = np.argsort(E_lanc)
E_lanc, x_lanc = E_lanc.T[idx][:e], x_lanc.T[:e]
#x_lanc = np.where(x_lanc>1e-13,x_lanc,0.0)
E_lanc, x_lanc 

In [ ]:
eig = np.linalg.eigh(H)
E_ed, x_ed = eig.eigenvalues[0:e],eig.eigenvectors.T[0:e]
E_ed, x_ed.shape

In [ ]:
mod_lanc = []
phase_lanc = []
mod_ed = []
phase_ed = []
stats_lanc = []
stats_ed = []
for x_E, x_L in zip(x_ed, x_lanc):
    assert len(x_E)==len(x_L)
    (mod_L, phase_L), stats_L = modphase(x_L)
    (mod_E, phase_E), stats_E = modphase(x_E)
    
    mod_lanc.append(mod_L)
    phase_lanc.append(mod_L)
    mod_ed.append(mod_E)
    phase_ed.append(phase_E)

    stats_lanc.append(stats_L)
    stats_ed.append(stats_E)




In [ ]:
title=r"J1:\;%.1f\quad J2:\;%.1f\qquad(%dx%d)"%(J1,J2,size[0],size[1])

In [ ]:
ms = 3

for mod_L, phase_L, stats_L, mod_E, phase_E, stats_E in zip(mod_lanc, phase_lanc, stats_lanc, mod_ed, phase_ed, stats_ed):
    fig,ax= plt.subplots(4,1, figsize=[20,15])

    ax[0].set_title(r"$Modulus\;and\;Phase\qquad %s$"%(title), fontsize=10)
    ax[0].set_xticks([])
    ax[0].set_ylabel(r"$Modulus$")
    ax[0].set_ylim(-0.00001,max(max(mod_E),max(mod_L))*9/8)
    ax[0].plot(mod_L, alpha=0.6, color='r', label=f"Lanczos")
    ax[0].plot(mod_E, alpha=0.6, label= f"ED")
    ax[0].legend()

    ax[1].set_xticks([])
    ax[1].set_yticks([-np.pi,-np.pi/2,0,np.pi/2,np.pi])
    ax[1].set_yticklabels([r"$-\pi$",r"$-\pi/2$",r"$0$",r"$\pi/2$",r"$\pi$"])
    ax[1].set_ylabel(r"$Phase \;ED$")
    ax[1].set_ylim(-np.pi-0.1, np.pi+0.1)
    ax[1].plot(phase_E, alpha=0.15, ls='', marker='o', ms=ms, color='r', label='ED')
    ax[1].legend(loc='upper right')

    ax[2].set_xlabel(r"$C_i$")
    ax[2].set_ylabel(r"$Phase \;Lanczos$")
    ax[2].set_ylim(-np.pi-0.1, np.pi+0.1)
    ax[2].set_yticks([-np.pi,-np.pi/2,0,np.pi/2,np.pi])
    ax[2].set_yticklabels([r"$-\pi$",r"$-\pi/2$",r"$0$",r"$\pi/2$",r"$\pi$"])
    ax[2].plot(phase_L, alpha=0.15, ls='', marker='o', ms=ms, label='Lanczos')
    ax[2].legend(loc='upper right')

    ax[3].set_xlabel(r"$Phase\;(radians)$")
    ax[3].set_ylabel(r"$Phase \;histogram$")
    ax[3].hist(phase_E, bins=500, range=(-np.pi, np.pi), color='r', density=True, alpha=0.9, label=f"ED  ({stats_E['type']})")
    ax[3].hist(phase_L, bins=500, range=(-np.pi, np.pi), color='b', density=True, alpha=0.7, label=f" ({stats_L['type']})")
    ax[3].text(0.8, 0.7, r"$\varphi_{ED}=%.2f \pm %.2f$"%(stats_E['phase']['mean'], stats_E['phase']['std']) + '\n' + r"$\varphi_{lanc}=%.2f \pm %.2f$"%(stats_L['phase']['mean'],stats_L['phase']['std']), 
                transform=ax[3].transAxes, fontsize=10,bbox=dict(facecolor="white", alpha=0.4))
    plt.show()
    plt.close()

In [ ]:
# Graficar histograma en coordenadas polares
counts_lanc, bin_edges = np.histogram(phase_E, bins= 500, range=(-np.pi, np.pi))
angles = (bin_edges[:-1] + bin_edges[1:]) /2

fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(111, polar=True)
# ax.bar(angles, counts_lanc, width=2*np.pi/500, align="center", edgecolor="k")
ax.plot(angles, counts_lanc, color="darkblue", linewidth=2)
ax.fill_between(angles, 0, counts_lanc, color="skyblue", alpha=0.5)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Ejemplo: datos discretos (fases)
phases = np.random.choice(np.linspace(-np.pi, np.pi, 12), size=500, p=np.random.dirichlet(np.ones(12)))

# Convertir a [0, 2pi]
phases = (phases + 2*np.pi) % (2*np.pi)

# Bins
num_bins = 36
counts, bin_edges = np.histogram(phases, bins=num_bins, range=(0, 2*np.pi))

# Ángulos (centro de cada bin)
angles = (bin_edges[:-1] + bin_edges[1:]) / 2

# Plot
fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(111, polar=True)

bars = ax.bar(
    angles, counts,
    width=2*np.pi/num_bins, 
    align="center",
    edgecolor="white",      # bordes blancos
    linewidth=1.2,
    color=plt.cm.viridis(counts / counts.max()),  # colormap bonito
)

# Opciones estéticas
ax.set_theta_zero_location("E")  # 0° en el norte
ax.set_theta_direction(-1)       # ángulos en sentido horario
ax.grid(alpha=0.3)               # cuadrícula tenue
ax.set_xticklabels([])           # ocultar radios

plt.show()

plt.show()

In [ ]:
%run ./SplitTraining_LRM.py

In [ ]:
from NN_module.models.ST_modules.CvT3_EDPPh import CvT3_EDPPh
from NN_module.models.ST_modules.CvT3_CvT3 import CvT3_CvT3
from NN_module.models.CvT3 import CvT3

model = CvT3_CvT3(
        lattice_size=(4,4),
        n_CP_blocks_list_1=[2],
        CTemb_channels_list_1=[32],
        CP_channels_list_1=[32],
        attn_heads_list_1=[4],
        kernel_1=(3,3),
        final_architecture_1=(5,),
        n_CP_blocks_list_2=[2],
        CTemb_channels_list_2=[32],
        CP_channels_list_2=[32],
        attn_heads_list_2=[4],
        kernel_2=(3,3),
        final_architecture_2=(5,),
        phasors=True,
        symm_Z2=False,
        trivial_Z2=True,
)

# model= CvT3(
#     lattice_size=(4,4),
#     n_CP_blocks_list=[2],
#     CTemb_channels_list=[32],
#     CP_channels_list=[32],
#     attn_heads_list=[4],
#     kernel=(3,3),
#     final_architecture=(5,),
#     symm_Z2=False,
#     trivial_Z2=False,
#         )

# model= CvT3_EDPPh(
#     lattice_size=(4,4),
#     n_CP_blocks_list=[2],
#     CTemb_channels_list=[32],
#     CP_channels_list=[32],
#     attn_heads_list=[4],
#     kernel=(3,3),
#     final_architecture=(5,),
#     edpph_channels=64,
#     symm_Z2=False,
#     trivial_Z2=False,
#         )
    


In [ ]:
rng=jax.random.PRNGKey(676)
size=(4,4)
B=4

x = jax.random.choice(rng,jnp.array([1,-1]),(B,*size,1)).reshape(B,size[0]*size[1])
x_inv = -1 * x


In [ ]:
params = model.init(rng, x)

y=model.apply(params, x)
y_inv=model.apply(params, x_inv)


# y=jnp.array([y.real,y.imag])
# y_inv=jnp.array([y_inv.real,y_inv.imag])
delta_y = y-y_inv

In [ ]:
a, b = y.real, y.imag
c, d = y_inv.real, y_inv.imag
z = jnp.exp(y) + jnp.exp(y_inv)
jnp.log(z)

In [ ]:
print(y)
print(y_inv)
b=jnp.array([1,-1])[:,None]
jax.nn.logsumexp(jnp.array([y, y_inv]), b=b, axis=0)

In [ ]:
jnp.log(jnp.exp(y)+jnp.exp(y_inv))

In [ ]:
mod, phase = np.loadtxt("/home/ihuarte/Escritorio/Ivan/NNs/prueba/J1J2Square_SplitTraining_CvT3_CNNPhasor/Size_4x4/ST_25P25Mx6/J1J2Square_SplitTraining_CvT3_CNNPhasor_xED_4x4_J1J2_0.5_1.0_XYZ_0.0_0.0_0.0_modphase_xED.txt")

_, ax = plt.subplots()

ax.plot(mod)

In [ ]:
xed= np.loadtxt("/home/ihuarte/Escritorio/Ivan/NNs/prueba/J1J2Square_SplitTraining_CvT3_CNNPhasor/Size_4x4/ST_25P25Mx6/J1J2Square_SplitTraining_CvT3_CNNPhasor_xED_4x4_J1J2_0.5_1.0_XYZ_0.0_0.0_0.0.txt", dtype=complex)
xed_filp = xed[::-1]


In [ ]:
def vmap_P_traslations(x, P_shifts):
    
    batched_traslations = lambda x, shift: jnp.roll(x, shift=shift, axis=(0,1))

    return jax.vmap(batched_traslations)(x, P_shifts)

def scan_P_traslations(x, P_shifts):

    def rollit(i, x_batch):
        x_b_trasl = jnp.roll(x_batch, shift=P_shifts[i], axis=(0,1))
        return i+1, x_b_trasl
    
    _, x_trasl = jax.lax.scan(rollit, init=0, xs=x)

    return x_trasl

def polyphase_components(x, strides):

    B, H, W, C = x.shape
    Hd, Wd = H // strides[0], W // strides[1], 
    xt = x.reshape(
        (B ,Hd ,strides[0],Wd,strides[1], C)
        ).transpose((0,1,3,4,2,5)).reshape(
            (B, Hd*Wd, *strides, C)
            ).transpose((0,3,2,1,4))

    shape = xt.shape
    poly_comp = xt.reshape(shape[0], shape[1]*shape[2], shape[3]*shape[4])

    return poly_comp

def get_maxnorm_indices(x, strides):
    """
    This function returns the traslation indices for a batched input of shape (B, H, W, C).
    It computes the polyphase components of a grid for each batch and channel, calculates 
    the L2 norm for each component and chooses the indices of the maximum value component.
    It works for both 1D and 2D inputs.
    Input:
        - x: (jnp.ArrayLike) Input data.
        - strides: (tuple) Strides for the next downsampling convolution.

    Returns:
        - x: Shifts (translations) for all batches and channels which makes the input 
             traslationaly equivariant.

    """
    _, H, W, _ = x.shape
    assert (H % strides[0]==0) & (W % strides[1]==0), f"`lattice_size` must be disible by `strides`. But they are {(H,W)} and {strides}"
    

    poly_comp = polyphase_components(x, strides)

    norm = jnp.linalg.norm(poly_comp, axis=-1)
    flat_idx = jnp.argmax(norm, axis=-1, keepdims=False)
    p, q = jnp.unravel_index(flat_idx, strides) 
    anchors = jnp.array([-p, -q]).T

    return anchors
     

def APS_equivariance_adapter(x, strides, mode = 'vmap'):

    P_shifts = get_maxnorm_indices(x, strides)

    if mode == 'scan':
        x = scan_P_traslations(x, P_shifts)
    elif mode == 'vmap':
        x = vmap_P_traslations(x, P_shifts)
    else:
        raise ValueError(f"No such mode: {mode}")

    return x






In [ ]:
x0 = jnp.zeros((1,4,4,1))
x0 = x0.at[0,1,0,0].set(10.0)  # activa un pixel en (1,0)

stride = (2,2)
shifts = get_maxnorm_indices(x0, stride)
print("shifts:", shifts)  # debería salir (-1,0)

x_hat = APS_equivariance_adapter(x0, stride)

In [ ]:
size = (8,8)
strides = (4,4)
assert (size[0] % strides[0]==0) & (size[1] % strides[1]==0), f"`lattice_size` must be disible by `strides`. But they are {size} and {strides}"
C = 1
B = 1

x = jnp.arange(size[0]*size[1]).reshape(*size)[None,:,:,None]
x = jnp.broadcast_to(x, (B, *size, C))


y = polyphase_equivariance_adapter(x, strides)
x.transpose((0,3,1,2)),y.transpose((0,3,1,2))

In [ ]:
x = jnp.zeros((1,8,8,3))
B, H, W, C = x.shape
strides=(2,2)
Hd, Wd = H // strides[0], W // strides[1]



# metemos valores grandes en la polifase (1,0)
coords = [(0,1), (0,3), (0,5), (0,7),
          (2,1), (2,3), (2,5), (2,7),
          (4,1), (4,3), (4,5), (4,7),
          (6,1), (6,3), (6,5), (6,7)]

for (i,j) in coords:
    x = x.at[0,i,j,:].set(7.0)

x.transpose((0,3,1,2))

In [ ]:
xt = x.reshape(
    (B ,Hd ,strides[0],Wd,strides[1], C)
    ).transpose((0,1,3,4,2,5)).reshape(
        (B, Hd*Wd, *strides, C)
        ).transpose((0,3,2,1,4))

shape = xt.shape
poly_comp = xt.reshape(shape[0], shape[1]*shape[2], shape[3]*shape[4])


In [ ]:
norm = jnp.linalg.norm(poly_comp, axis=-1)
flat_idx = jnp.argmax(norm, axis=-1)
p, q = jnp.unravel_index(flat_idx, strides) 
anchors =jnp.array([-p, -q]).T
anchors

In [ ]:
ST_setup={
    "segments":[[75,25],[50,50],[100]],
    "mode":[["P","M"], ["P","M"],["P"]],
    "repeat_segment":[5,4,1]
        }

raw_segments = [[seg]*rep for seg, rep in zip(ST_setup["segments"],ST_setup["repeat_segment"])]
segments = [seg for i in range(len(raw_segments)) for seg in raw_segments[i]]
raw_modes = [[mode]*rep for mode, rep in zip(ST_setup["mode"],ST_setup["repeat_segment"])]
modes = [seg for i in range(len(raw_segments)) for seg in raw_modes[i]]

segments = []
modes = []
s, m, r = ST_setup.values()
for seg, mode, repeats in zip(s,m,r):
    segments += [seg] * repeats
    modes += [mode] *repeats

segments, modes

In [ ]:
label = "ST"
s, m, r = ST_setup.values()
for seg, mode, repeats in zip(s, m, r):
    label += "_"
    for s, m in zip(seg, mode):
        label += f"{s}{m}"
    label += f"x{repeats}"

label

In [ ]:
B, H, W, C = x.shape
Hd, Wd = H // strides[0], W // strides[1], 
# print(f"(B, H, W, C) = {x.shape}")
# print(f"strides: {strides}")
# print(f"Hd, Wd = {Hd}, {Wd}")
x = x.reshape(
    (B, Wd, strides[1], Hd, strides[0], C),
    order='C').transpose((0, 1, 3, 2, 4, 5)
    ).reshape(B, Hd*Wd, *strides, C)
x.transpose(0,2,3,1,4)

In [ ]:
poly = polyphase_components(x, strides)
shape = poly.shape
poly = poly.reshape(*shape[0:-2], shape[-2]*shape[-1])

norm = jnp.linalg.norm(poly, axis=-1)
-jnp.argmax(norm, axis=-1)


In [ ]:
x.squeeze()

In [ ]:
size = (4,4)
strides = (2,2)
assert (size[0] % strides[0]==0) & (size[1] % strides[1]==0), f"`lattice_size` must be disible by `strides`. But they are {size} and {strides}"
C = 8
B = 1

# key=jax.random.PRNGKey(977686)
# x_broad = jax.random.uniform(key, shape=(B,*size,C))
# fun = jax.jit(polyphase_equivariance_adapter(strides))
# fun(jax.random.uniform(key, shape=(B,*size,C)))

# %timeit -n 10000 -r 100 fun(jax.random.uniform(key, shape=(B,*size,C)))

In [ ]:
import itertools
def get_channel_indices(H, W, C):
    print(H,W,C)
    assert (H*W)**C < 3e5, f"Too much channels or lattice size"
    
    grid_tras_ind = jnp.array([(i,j) for i in range(H) for j in range(W)])
    combo_idx = jnp.array(list(itertools.product(range(H*W), repeat=C)))
    return grid_tras_ind[combo_idx]

H = 2
W = 2
C = 5
get_channel_indices(H,W,C).shape[-2]

In [ ]:
(H*W)**C > 3e5

In [ ]:
3e5

In [ ]:

_, ax = plt.subplots()
ax.hist(phase,rwidth=0.1)
ax.set_xlabel(r"$\varphi$")
ax.set_ylabel(r"$P(\varphi)$")

In [ ]:
set(phase),bins

In [ ]:
oxa=OxalateJKGamma(
    [4,5], 
    [0.2, 9.0, 72.0],
    **{'bc': 'periodic', 'order': 'default_2'}
)
E_ED, x_ED = Runner(oxa.cm).exact_energy_lanczos(k=5, eigenstates=True)

In [ ]:
art_path='/home/ihuarte/Escritorio/Ivan/NNs/Simulations/Oxalate_SplitTraining_ViT_CNN/Size_4x5/sweeps_0/Oxalate_SplitTraining_ViT_CNN_results_4x5_strength_0.2_theta_9.0_phi_72.0_channels_32_kernel_3x3_n_ffn_lay_1.json'

with open(art_path,'r') as f:
    artifact = json.load(f)

In [ ]:
x_ED = np.loadtxt("/home/ihuarte/Escritorio/Ivan/NNs/prueba/JKGamma/Split_oxalate_CNN/Oxalate_size_4x4/sweeps_20/Oxalate_xED_4x4_strength_0.2_theta_9.0_phi_72.0_ED.txt", dtype=complex)
x_vs = np.array(vstate.to_array())

In [ ]:



mod_vs, phase_vs, stats_vs = modphase(x_vs)
mod_ED, phase_ED, stats_ED = modphase(x_ED)



In [ ]:
a=(383,339)
print(f"|| {a} ||")

In [ ]:
import matplotlib.transforms as mtransforms
_,ax= plt.subplots(4,1, figsize=[15,10])

ax[0].set_title(r"$Modulus\;and\;Phase\qquad Oxalate\;size\;%d x %d \qquad a=%.1f\;\;\theta=%.1f \;\; \phi=%.1f$"%(4,4,0.2,9.0,72.0))
ax[0].set_xticks([])
ax[0].set_ylabel(r"$Modulus$")
ax[0].set_ylim(-0.01,max(max(mod_ED),max(mod_vs))*9/8)
ax[0].plot(mod_vs, alpha=0.6, color='r', label='vstate')
ax[0].plot(mod_ED, alpha=0.6, label='ED')
ax[0].legend()

ax[1].set_xticks([])
ax[1].set_yticks([-np.pi,-np.pi/2,0,np.pi/2,np.pi])
ax[1].set_yticklabels([r"$-\pi$",r"$-\pi/2$",r"$0$",r"$\pi/2$",r"$\pi$"])
ax[1].set_ylabel(r"$Phase \;vstate$")
ax[1].set_ylim(-np.pi-0.1, np.pi+0.1)
ax[1].plot(phase_vs, alpha=0.25, ls='', marker='o', ms=0.9, color='r', label='vstate')
ax[1].legend(loc='upper right')

ax[2].set_xlabel(r"$C_i$")
ax[2].set_ylabel(r"$Phase \;ED$")
ax[2].set_ylim(-np.pi-0.1, np.pi+0.1)
ax[2].set_yticks([-np.pi,-np.pi/2,0,np.pi/2,np.pi])
ax[2].set_yticklabels([r"$-\pi$",r"$-\pi/2$",r"$0$",r"$\pi/2$",r"$\pi$"])
ax[2].plot(phase_ED, alpha=0.25, ls='', marker='o', ms=0.9, label='ED')
ax[2].legend(loc='upper right')

ax[3].set_xlabel(r"$Phase\;(radians)$")
ax[3].set_ylabel(r"$Phase \;histogram$")
ax[3].hist(phase_ED, bins=1000, range=(-np.pi, np.pi), density=True, alpha=0.7, label='ED')
ax[3].hist(phase_vs, bins=1000, range=(-np.pi, np.pi), color='r', density=True, alpha=0.7, label='vstate')


transform = mtransforms.blended_transform_factory(ax[3].transData, ax[3].transAxes)
if stats_ED['peaks'] is not None:
    for peak in stats_ED['peaks']['values']:
        ax[3].text(peak-0.1, 0.9, r"%.2f"%peak, color='b', transform=transform, fontsize=8, alpha=0.7)

if stats_vs['peaks'] is not None:
    for peak in stats_vs['peaks']['values']:
        ax[3].text(peak-0.1, 0.9, r"%.2f"%peak, color='b', transform=transform, fontsize=8, alpha=0.7)
ax[3].legend()
plt.tight_layout()
plt.savefig(write_folder + files[j] + ".jpeg", dpi=600, bbox_inches="tight")

In [ ]:
_, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(lr_schedule, marker='o',ls=None, label='Learning Rate Schedule')
ax.set_xlabel('Epochs')
ax.set_ylabel('Learning Rate')

In [ ]:
from VA_project.lattice.lattice import Chain,Square
from VA_project.model.cm import GeneralNeighborCoupling

size=[4,5]

fields=[(0., 'X'), (0.5, 'Z'), (0.8, 'Y')]
couplings=[(0.,'ZZ','NN'),(2.,'YY','NN2')]

chain= Square(*size,bc='periodic', order= 'default_1')
ZZYY= GeneralNeighborCoupling(chain, fields, couplings)
chain.plot_lattice(1)

In [ ]:
from NN_module.NN_utils import traslations_2D,traslations_2D_scan,traslations_2D_vmap

In [ ]:
a=jnp.arange(20)
size=[4,5]
token_size=[2,1]
sub_lat=[size[0]//token_size[0], size[1]//token_size[1]]
# a.reshape((sub_lat[1],token_size[1],sub_lat[0],token_size[0]),
#                   order='C').transpose((0, 2, 1, 3)).reshape(-1,token_size[0]*token_size[1]).squeeze()
#traslations_2D_scan(a,size).reshape(-1,*size)
traslations_2D(a,size, memory=False)#.reshape(-1,*size)



In [ ]:
import sys
from pathlib import Path
sys.path.append(str("/home/ihuarte/Escritorio/Ivan/ATMOS_VA/VA_project/src"))

from VA_project.model.model import OxalateJKGamma
from VA_project.engine.runners import Runner
from NN_module.sim_utils import (
    load_vstate
)
files=[
    "/home/ihuarte/Escritorio/Ivan/NNs/Simulations/ViT_2D/Oxalate_size_4x4/targets/Oxalate_results_4x4_strength_0.2_theta_54.0_phi_0.0_b_2x2_Demb_128_heads_2_blocks_2_ffn_lay_2.json",
    "/home/ihuarte/Escritorio/Ivan/NNs/Simulations/ViT_2D/Oxalate_size_4x4/targets/Oxalate_results_4x4_strength_0.2_theta_54.0_phi_0.0_b_2x2_Demb_64_heads_8_blocks_2_ffn_lay_4.json",
    "/home/ihuarte/Escritorio/Ivan/NNs/Simulations/ViT_2D/Oxalate_size_4x4/targets/Oxalate_results_4x4_strength_0.2_theta_54.0_phi_0.0_b_2x2_Demb_64_heads_2_blocks_2_ffn_lay_4.json"
]

size=[4,4]
strength=0.2
theta=54.
phi=0.
kwargs_lattice={
    'bc':'periodic',
    'order':'default_2'
}

oxa=OxalateJKGamma(
            size, 
            [strength, theta, phi],
            **kwargs_lattice
        )
H = Runner(oxa.cm).build_hamiltonian()
E_ED, x_ED = Runner(oxa.cm).exact_energy_lanczos(eigenstates=True, k=150)
idx=np.argsort(E_ED)
E_ED=E_ED[idx]
x_ED=x_ED.T[idx,:]

In [ ]:
i=2
path_artifact= files[i]

with open(path_artifact,'r') as f:
    artifact = json.load(f)

vstate=load_vstate(artifact).to_array()
vstate=np.array(vstate, dtype=np.complex128)


In [ ]:
c=np.sum(x_ED.conjugate()*vstate, axis=1)
P=np.abs(c)**2
sum(P)

In [ ]:
E_best=artifact['results']['E_best']
e=-np.inf; idx=0
while e<E_best:
    e=E_ED[idx]
    idx+=1

idx, E_ED[idx],E_ED[idx-1]

In [ ]:
P_sum=sum(P)

_,ax=plt.subplots()
ax.set_title(f"Projections #{i}")
ax.set_xlabel("Eigenstates")
ax.set_ylabel("Probability")
ax.bar(range(len(P)), P, label=r"$\Sigma_i\; P_i$")
ax.vlines(idx, 0,max(P), color='r', ls='--', label=r"$\sim E_{sim}$")
ax.legend(fontsize=8)
ax.text(0.5, 0.8, r"$\Sigma_i\; P_i = %.4f$"%P_sum+"\n"+r"$E_1=%.3f$"%E_ED[0]+"\n"+r"$E_{150}=%.3f$"%(E_ED[-1])+"\n"+r"$E_{sim}=%.3f$"%E_best, transform=ax.transAxes, fontsize=10, bbox=dict(facecolor="white", alpha=0.4))
#ax.set_yscale('log')

In [ ]:
a=False
b=True
not a and b

# Learning Rate Scheduler

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp
from NN_module.NN_utils import scheduler_initializer

In [ ]:

schedule={
    'epochs': 500,
    'cos_exp_scheduler':{
        "lr0":0.6,
        "decay_exp":1.,
        "cosine_cycles":2.,
        "n":0.1,
        "lr_min": 0.01
    }
}

lr_sch=scheduler_initializer('cos_exp_scheduler', schedule)


# recta = lambda x: (lr_min-n)/epochs*x+n
# y_recta=recta(jnp.arange(0,epochs))

y=lr_sch(jnp.arange(0,schedule['epochs']))

_,ax= plt.subplots(1,1,figsize=(10,5))
ax.plot(y, label="exp decay")
#ax.plot(np.arange(epochs),y_recta, ls= '--', color='r')
ax.set_xlabel("step")
ax.set_ylabel("learning rate")
#ax.set_ylim(0, 1.05)
ax.legend()
ax.grid()

In [ ]:
21388438632/(1024)**3

In [ ]:
import jax.numpy as jnp
import jax
a= jnp.ones((4,4), dtype=complex)/10
b= jnp.ones((4,4) ,dtype=complex)


stack=jnp.array([a,b]).transpose(1,0,2)
z2_sym=jax.nn.logsumexp(
    stack,
    b=jnp.array([1., -1.])[None,:,None],
    axis=1
)
z2_2d_symm=jax.nn.logsumexp(
    z2_sym,
    axis=0
)

In [ ]:
from flax.serialization import to_bytes, from_bytes
import msgpack
file="/home/ihuarte/Escritorio/Ivan/NNs/Simulations/ViT_2D/Oxalate_size_4x4/Oxalate_vstate_params_4x4_strength_0.2_theta_90.0_phi_115.2_b_2x1_Demb_64_heads_4_blocks_2_ffn_lay_4.msgpack"

In [ ]:
with open(file, "rb") as f:
    bytes_data = f.read()

# Decodifica con msgpack sin un schema de Flax
unpacked = msgpack.unpackb(bytes_data, raw=False)

import pprint
pprint.pprint(unpacked)


In [ ]:
_, ax= plt.subplots()
x=np.arange(50,300)
ax.set_ylim(0,0.1)
ax.set_xlim(0,300)
ax.plot(x,0.05*0.987**(x-50))

In [ ]:
import numpy as np
file="/home/ihuarte/Escritorio/Ivan/NNs/Simulations/pruebas_Ising_LRM/LRM/ViT/Oxalate_xED_4x5_strength_0.2_theta_0_phi_0.txt"
x_ED= np.loadtxt(file, dtype=complex)
x_ED

In [ ]:
jnp.angle(x_ED)

In [ ]:
from NN_module.sim_utils import load_vstate
import json
file="/home/ihuarte/Escritorio/Ivan/NNs/Simulations/pruebas_Ising_LRM/LRM/ViT/Oxalate_results_4x5_strength_0.2_theta_0_phi_0_XZ_-1.0_0.001_J_1.5_alpha_2.5_b_2x1_Demb_32_heads_2_blocks_2_ffn_lay_2.json"

In [ ]:
with open(file,'rb') as f:
    conf=json.load(f)

vstate=load_vstate(conf)